# CKD Deepfake — Colab Setup

Run this notebook at the start of every Colab session.

**Order:**
1. Mount Google Drive (persistent storage for datasets, checkpoints, results)
2. Clone/pull repository from GitHub
3. Install dependencies
4. Copy hot data from Drive to local disk (fast I/O during training)
5. Verify GPU + config loader

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/CKD_Thesis'
!ls {DRIVE_ROOT}

## 2. Clone Repository

Replace `GITHUB_USER` with your GitHub username before the first run.

In [ ]:
import os

GITHUB_USER = 'USERNAME'  # TODO: replace
REPO_NAME = 'ckd-deepfake'
REPO_URL = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

!git log -1 --oneline

## 3. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

## 4. Copy Hot Data to Local Disk

Drive I/O is slow during training. Copy extracted faces + split CSVs + soft labels
to `/content/ckd_local/` for ~10-20x faster data loading.

Only copies if not already present (idempotent).

In [ ]:
import os, shutil, time

LOCAL_ROOT = '/content/ckd_local'
os.makedirs(LOCAL_ROOT, exist_ok=True)

SUBDIRS = ['faces', 'splits', 'soft_labels']

for sub in SUBDIRS:
    src = os.path.join(DRIVE_ROOT, sub)
    dst = os.path.join(LOCAL_ROOT, sub)
    if not os.path.isdir(src):
        print(f'[skip] {src} does not exist yet')
        continue
    if os.path.isdir(dst):
        print(f'[skip] {dst} already copied')
        continue
    t0 = time.time()
    print(f'[copy] {src} -> {dst}')
    shutil.copytree(src, dst)
    print(f'  done in {time.time() - t0:.1f}s')

!du -sh {LOCAL_ROOT}/* 2>/dev/null || echo '(empty)'

## 5. Verify GPU and Config Loader

In [ ]:
import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Runtime > Change runtime type > GPU.')

In [ ]:
from src.utils.config import load_config

cfg = load_config('configs/default.yaml')
print('Config loaded successfully.')
print(f'  student: {cfg["student"]["name"]}')
print(f'  teachers: {[t["name"] for t in cfg["teacher"]["models"]]}')
print(f'  drive_root: {cfg["paths"]["drive_root"]}')